# 03 — Revenue and Retention Analysis

## Brazilian E-Commerce Analytics Portfolio

This notebook analyzes business performance using the Olist e-commerce dataset.

### Objectives

1. Build order-level and customer-level analytical tables.
2. Calculate business KPIs.
3. Analyze monthly order volume and revenue.
4. Compare revenue by state, product category, seller, and payment type.
5. Measure repeat-customer behavior.
6. Build monthly customer cohorts and retention matrices.
7. Export processed business-analysis tables, figures, and a Markdown report.

### Main questions answered by this notebook

- How did revenue evolve over time?
- What are the highest-revenue product categories?
- Which customer states generate the most revenue?
- Which sellers generate the most item revenue?
- What is the average order value?
- What percentage of customers purchase more than once?
- How does customer retention behave by monthly cohort?

> Revenue values are descriptive analytical values derived from the dataset. They should not be interpreted as audited financial statements.

## Methodology workflow

```text
Raw Olist relational datasets
        ↓
Build order-level table
        ↓
Build customer-level table
        ↓
Calculate revenue, order, and customer KPIs
        ↓
Analyze monthly revenue and order trends
        ↓
Analyze revenue by state, category, seller, and payment type
        ↓
Build customer purchase cohorts
        ↓
Calculate repeat-purchase and retention rates
        ↓
Export figures, processed tables, and business report
```

The corresponding workflow image can be stored as:

```text
workflows/03_revenue_and_retention_workflow.png
```

## 1. Import libraries and configure the environment

In [ ]:
from pathlib import Path
from typing import Dict, Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

## 2. Resolve project paths and locate the raw data

In [ ]:
def find_local_raw_data(start_path: Path) -> Path | None:
    """Find a local data/raw directory containing the Olist CSV files."""
    required_file = "olist_orders_dataset.csv"
    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:
        raw_dir = candidate / "data" / "raw"
        if (raw_dir / required_file).exists():
            return raw_dir

    return None


PROJECT_ROOT = Path.cwd()
LOCAL_RAW_DATA_DIR = find_local_raw_data(PROJECT_ROOT)

if LOCAL_RAW_DATA_DIR is not None:
    RAW_DATA_DIR = LOCAL_RAW_DATA_DIR
    PROJECT_ROOT = RAW_DATA_DIR.parents[1]
    print("Using local data/raw directory.")
else:
    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError(
            "kagglehub is not installed and local data/raw was not found. "
            "Install it with: pip install kagglehub"
        ) from exc

    RAW_DATA_DIR = Path(
        kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    )
    print("Using KaggleHub dataset directory.")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:       {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data:     {PROCESSED_DATA_DIR}")
print(f"Reports directory:  {REPORTS_DIR}")
print(f"Figures directory:  {FIGURES_DIR}")

## 3. Load the Olist datasets

In [ ]:
FILE_NAMES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

DATE_COLUMNS = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": ["shipping_limit_date"],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}


def load_dataset(file_path: Path, date_columns: Iterable[str] | None = None) -> pd.DataFrame:
    """Load one CSV file and parse date columns when available."""
    dataframe = pd.read_csv(file_path, low_memory=False)

    if date_columns:
        for column in date_columns:
            if column in dataframe.columns:
                dataframe[column] = pd.to_datetime(dataframe[column], errors="coerce")

    return dataframe


missing_files = [
    file_name for file_name in FILE_NAMES.values()
    if not (RAW_DATA_DIR / file_name).exists()
]

if missing_files:
    missing_text = "\n".join(f"- {name}" for name in missing_files)
    raise FileNotFoundError(f"Missing required files:\n{missing_text}")

datasets: Dict[str, pd.DataFrame] = {}

for dataset_name, file_name in FILE_NAMES.items():
    datasets[dataset_name] = load_dataset(
        RAW_DATA_DIR / file_name,
        DATE_COLUMNS.get(dataset_name),
    )

for dataset_name, dataframe in datasets.items():
    print(f"{dataset_name:<22} {dataframe.shape[0]:>8,} rows × {dataframe.shape[1]:>2} columns")

## 4. Build analytical tables

In [ ]:
def build_analytical_tables(datasets: Dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Create order-level, customer-level, and item-product analytical tables."""
    customers = datasets["customers"].copy()
    orders = datasets["orders"].copy()
    order_items = datasets["order_items"].copy()
    payments = datasets["payments"].copy()
    reviews = datasets["reviews"].copy()
    products = datasets["products"].copy()
    category_translation = datasets["category_translation"].copy()

    items_with_products = (
        order_items
        .merge(products, on="product_id", how="left")
        .merge(category_translation, on="product_category_name", how="left")
    )
    items_with_products["product_category_name_english"] = (
        items_with_products["product_category_name_english"].fillna("unknown")
    )

    item_summary = (
        items_with_products
        .groupby("order_id", as_index=False)
        .agg(
            item_count=("order_item_id", "count"),
            unique_products=("product_id", "nunique"),
            unique_sellers=("seller_id", "nunique"),
            total_item_value=("price", "sum"),
            total_freight_value=("freight_value", "sum"),
        )
    )

    category_by_order = (
        items_with_products
        .groupby(["order_id", "product_category_name_english"], as_index=False)
        .agg(category_item_value=("price", "sum"))
        .sort_values(["order_id", "category_item_value"], ascending=[True, False])
        .drop_duplicates(subset=["order_id"], keep="first")
        .rename(columns={"product_category_name_english": "primary_product_category"})
        .loc[:, ["order_id", "primary_product_category"]]
    )

    payment_summary = (
        payments
        .groupby("order_id", as_index=False)
        .agg(
            payment_records=("payment_sequential", "count"),
            payment_types=("payment_type", "nunique"),
            max_installments=("payment_installments", "max"),
            total_payment_value=("payment_value", "sum"),
        )
    )

    main_payment_type = (
        payments
        .groupby(["order_id", "payment_type"], as_index=False)
        .size()
        .sort_values(["order_id", "size", "payment_type"], ascending=[True, False, True])
        .drop_duplicates(subset=["order_id"], keep="first")
        .rename(columns={"payment_type": "main_payment_type"})
        .loc[:, ["order_id", "main_payment_type"]]
    )

    review_summary = (
        reviews
        .assign(
            has_review_comment=lambda dataframe: (
                dataframe["review_comment_message"].notna()
                | dataframe["review_comment_title"].notna()
            )
        )
        .groupby("order_id", as_index=False)
        .agg(
            review_count=("review_id", "count"),
            average_review_score=("review_score", "mean"),
            has_review_comment=("has_review_comment", "max"),
        )
    )

    order_level = (
        orders
        .merge(customers, on="customer_id", how="left")
        .merge(item_summary, on="order_id", how="left")
        .merge(category_by_order, on="order_id", how="left")
        .merge(payment_summary, on="order_id", how="left")
        .merge(main_payment_type, on="order_id", how="left")
        .merge(review_summary, on="order_id", how="left")
    )

    order_level["order_purchase_month"] = order_level["order_purchase_timestamp"].dt.to_period("M").astype("string")
    order_level["actual_delivery_days"] = (
        order_level["order_delivered_customer_date"] - order_level["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400
    order_level["delivery_delay_days"] = (
        order_level["order_delivered_customer_date"] - order_level["order_estimated_delivery_date"]
    ).dt.total_seconds() / 86400
    order_level["is_delivered"] = order_level["order_status"].eq("delivered")
    order_level["is_late_delivery"] = np.where(
        order_level["order_delivered_customer_date"].notna(),
        order_level["delivery_delay_days"] > 0,
        np.nan,
    )
    order_level["has_review_comment"] = order_level["has_review_comment"].fillna(False).astype(bool)

    customer_level = (
        order_level
        .groupby("customer_unique_id", as_index=False)
        .agg(
            customer_state=("customer_state", "first"),
            customer_city=("customer_city", "first"),
            first_purchase_timestamp=("order_purchase_timestamp", "min"),
            last_purchase_timestamp=("order_purchase_timestamp", "max"),
            total_orders=("order_id", "nunique"),
            delivered_orders=("is_delivered", "sum"),
            total_items=("item_count", "sum"),
            total_item_value=("total_item_value", "sum"),
            total_freight_value=("total_freight_value", "sum"),
            total_payment_value=("total_payment_value", "sum"),
            average_review_score=("average_review_score", "mean"),
        )
    )
    customer_level["average_order_value"] = (
        customer_level["total_payment_value"] / customer_level["total_orders"].replace({0: np.nan})
    )
    customer_level["is_repeat_customer"] = customer_level["total_orders"] > 1
    customer_level["customer_lifetime_days"] = (
        customer_level["last_purchase_timestamp"] - customer_level["first_purchase_timestamp"]
    ).dt.days

    return order_level, customer_level, items_with_products


order_level, customer_level, items_with_products = build_analytical_tables(datasets)

print(f"Order-level table:    {order_level.shape[0]:,} rows × {order_level.shape[1]} columns")
print(f"Customer-level table: {customer_level.shape[0]:,} rows × {customer_level.shape[1]} columns")
print(f"Item-product table:   {items_with_products.shape[0]:,} rows × {items_with_products.shape[1]} columns")

## 5. Business KPI summary

The project uses two complementary monetary definitions:

- `total_item_value`: sum of item prices, similar to merchandise value before freight;
- `total_payment_value`: sum of customer payments, including freight and payment records.

Most business summaries below use `total_payment_value` as the customer-paid transaction value.

In [ ]:
orders_with_payment = order_level.loc[order_level["total_payment_value"].notna()].copy()
delivered_orders = order_level.loc[order_level["is_delivered"]].copy()

business_kpi_summary = pd.DataFrame([
    {"metric": "total_orders", "value": int(order_level["order_id"].nunique())},
    {"metric": "delivered_orders", "value": int(order_level["is_delivered"].sum())},
    {"metric": "unique_customers", "value": int(customer_level["customer_unique_id"].nunique())},
    {"metric": "repeat_customers", "value": int(customer_level["is_repeat_customer"].sum())},
    {"metric": "repeat_customer_rate", "value": round(customer_level["is_repeat_customer"].mean(), 4)},
    {"metric": "total_payment_value", "value": round(orders_with_payment["total_payment_value"].sum(), 2)},
    {"metric": "total_item_value", "value": round(order_level["total_item_value"].sum(), 2)},
    {"metric": "average_order_value", "value": round(orders_with_payment["total_payment_value"].mean(), 2)},
    {"metric": "median_order_value", "value": round(orders_with_payment["total_payment_value"].median(), 2)},
    {"metric": "average_review_score", "value": round(order_level["average_review_score"].mean(), 3)},
    {"metric": "late_delivery_rate", "value": round(delivered_orders["is_late_delivery"].astype(float).mean(), 4)},
])

business_kpi_summary

## 6. Monthly revenue and order volume

In [ ]:
monthly_performance = (
    order_level
    .dropna(subset=["order_purchase_month"])
    .groupby("order_purchase_month", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        unique_customers=("customer_unique_id", "nunique"),
        payment_value=("total_payment_value", "sum"),
        item_value=("total_item_value", "sum"),
        freight_value=("total_freight_value", "sum"),
        average_order_value=("total_payment_value", "mean"),
        average_review_score=("average_review_score", "mean"),
    )
    .sort_values("order_purchase_month")
)

monthly_performance.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_performance["order_purchase_month"], monthly_performance["payment_value"], marker="o")
plt.title("Monthly Payment Value")
plt.xlabel("Purchase month")
plt.ylabel("Payment value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
monthly_revenue_figure_path = FIGURES_DIR / "monthly_payment_value.png"
plt.savefig(monthly_revenue_figure_path, dpi=150)
plt.show()
print(f"Saved figure: {monthly_revenue_figure_path.relative_to(PROJECT_ROOT)}")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_performance["order_purchase_month"], monthly_performance["order_count"], marker="o")
plt.title("Monthly Order Count")
plt.xlabel("Purchase month")
plt.ylabel("Number of orders")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
monthly_orders_figure_path = FIGURES_DIR / "monthly_order_count.png"
plt.savefig(monthly_orders_figure_path, dpi=150)
plt.show()
print(f"Saved figure: {monthly_orders_figure_path.relative_to(PROJECT_ROOT)}")

## 7. Revenue by customer state

In [ ]:
revenue_by_state = (
    order_level
    .groupby("customer_state", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        unique_customers=("customer_unique_id", "nunique"),
        payment_value=("total_payment_value", "sum"),
        average_order_value=("total_payment_value", "mean"),
        average_review_score=("average_review_score", "mean"),
    )
    .sort_values("payment_value", ascending=False)
)

revenue_by_state.head(15)

In [ ]:
top_states_by_revenue = revenue_by_state.head(15)
plt.figure(figsize=(10, 5))
plt.bar(top_states_by_revenue["customer_state"], top_states_by_revenue["payment_value"])
plt.title("Top Customer States by Payment Value")
plt.xlabel("Customer state")
plt.ylabel("Payment value")
plt.tight_layout()
state_revenue_figure_path = FIGURES_DIR / "top_customer_states_by_payment_value.png"
plt.savefig(state_revenue_figure_path, dpi=150)
plt.show()
print(f"Saved figure: {state_revenue_figure_path.relative_to(PROJECT_ROOT)}")

## 8. Revenue by product category

In [ ]:
category_revenue = (
    items_with_products
    .groupby("product_category_name_english", as_index=False)
    .agg(
        item_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count"),
        order_count=("order_id", "nunique"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
    )
    .sort_values("item_revenue", ascending=False)
)

category_revenue.head(15)

In [ ]:
top_categories = category_revenue.head(15)
plt.figure(figsize=(11, 6))
plt.bar(top_categories["product_category_name_english"], top_categories["item_revenue"])
plt.title("Top Product Categories by Item Revenue")
plt.xlabel("Product category")
plt.ylabel("Item revenue")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
category_revenue_figure_path = FIGURES_DIR / "top_product_categories_by_item_revenue.png"
plt.savefig(category_revenue_figure_path, dpi=150)
plt.show()
print(f"Saved figure: {category_revenue_figure_path.relative_to(PROJECT_ROOT)}")

## 9. Seller revenue analysis

In [ ]:
seller_revenue = (
    items_with_products
    .merge(datasets["sellers"], on="seller_id", how="left")
    .groupby(["seller_id", "seller_state"], as_index=False)
    .agg(
        item_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count"),
        order_count=("order_id", "nunique"),
        unique_products=("product_id", "nunique"),
    )
    .sort_values("item_revenue", ascending=False)
)

seller_revenue.head(20)

## 10. Payment behavior

In [ ]:
payment_behavior = (
    order_level
    .groupby("main_payment_type", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        payment_value=("total_payment_value", "sum"),
        average_order_value=("total_payment_value", "mean"),
        median_order_value=("total_payment_value", "median"),
        average_installments=("max_installments", "mean"),
    )
    .sort_values("payment_value", ascending=False)
)

payment_behavior

## 11. Repeat-customer analysis

In [ ]:
repeat_customer_summary = (
    customer_level
    .assign(customer_type=np.where(customer_level["is_repeat_customer"], "repeat_customer", "one_time_customer"))
    .groupby("customer_type", as_index=False)
    .agg(
        customer_count=("customer_unique_id", "nunique"),
        total_orders=("total_orders", "sum"),
        payment_value=("total_payment_value", "sum"),
        average_customer_value=("total_payment_value", "mean"),
        median_customer_value=("total_payment_value", "median"),
        average_order_value=("average_order_value", "mean"),
    )
)
repeat_customer_summary["customer_percentage"] = repeat_customer_summary["customer_count"] / repeat_customer_summary["customer_count"].sum()

repeat_customer_summary

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(repeat_customer_summary["customer_type"], repeat_customer_summary["customer_count"])
plt.title("One-Time vs Repeat Customers")
plt.xlabel("Customer type")
plt.ylabel("Number of customers")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
repeat_customers_figure_path = FIGURES_DIR / "one_time_vs_repeat_customers.png"
plt.savefig(repeat_customers_figure_path, dpi=150)
plt.show()
print(f"Saved figure: {repeat_customers_figure_path.relative_to(PROJECT_ROOT)}")

## 12. Monthly customer cohorts and retention

In [ ]:
cohort_source = order_level.loc[order_level["order_purchase_timestamp"].notna()].copy()
cohort_source["order_month"] = cohort_source["order_purchase_timestamp"].dt.to_period("M")

first_purchase = (
    cohort_source
    .groupby("customer_unique_id", as_index=False)
    .agg(cohort_month=("order_month", "min"))
)

cohort_source = cohort_source.merge(first_purchase, on="customer_unique_id", how="left")
cohort_source["months_since_first_purchase"] = (
    (cohort_source["order_month"].dt.year - cohort_source["cohort_month"].dt.year) * 12
    + (cohort_source["order_month"].dt.month - cohort_source["cohort_month"].dt.month)
)

cohort_counts = (
    cohort_source
    .groupby(["cohort_month", "months_since_first_purchase"], as_index=False)
    .agg(active_customers=("customer_unique_id", "nunique"))
)

cohort_sizes = (
    cohort_counts
    .loc[cohort_counts["months_since_first_purchase"].eq(0)]
    .loc[:, ["cohort_month", "active_customers"]]
    .rename(columns={"active_customers": "cohort_size"})
)

cohort_retention = cohort_counts.merge(cohort_sizes, on="cohort_month", how="left")
cohort_retention["retention_rate"] = cohort_retention["active_customers"] / cohort_retention["cohort_size"]
cohort_retention["cohort_month"] = cohort_retention["cohort_month"].astype(str)

retention_matrix = (
    cohort_retention
    .pivot(index="cohort_month", columns="months_since_first_purchase", values="retention_rate")
    .sort_index()
)

cohort_retention.head(20)

In [ ]:
plt.figure(figsize=(12, 7))
plt.imshow(retention_matrix.fillna(0), aspect="auto")
plt.title("Monthly Customer Retention Matrix")
plt.xlabel("Months since first purchase")
plt.ylabel("Cohort month")
plt.xticks(ticks=np.arange(retention_matrix.shape[1]), labels=retention_matrix.columns.astype(str))
plt.yticks(ticks=np.arange(retention_matrix.shape[0]), labels=retention_matrix.index.astype(str))
plt.colorbar(label="Retention rate")
plt.tight_layout()
retention_matrix_figure_path = FIGURES_DIR / "monthly_customer_retention_matrix.png"
plt.savefig(retention_matrix_figure_path, dpi=150)
plt.show()
print(f"Saved figure: {retention_matrix_figure_path.relative_to(PROJECT_ROOT)}")

## 13. Export business-analysis artifacts

In [ ]:
business_outputs = {
    "order_level_summary.csv": order_level,
    "customer_level_summary.csv": customer_level,
    "business_kpi_summary.csv": business_kpi_summary,
    "monthly_performance.csv": monthly_performance,
    "revenue_by_state.csv": revenue_by_state,
    "category_revenue.csv": category_revenue,
    "seller_revenue.csv": seller_revenue,
    "payment_behavior.csv": payment_behavior,
    "repeat_customer_summary.csv": repeat_customer_summary,
    "cohort_retention.csv": cohort_retention,
    "cohort_retention_matrix.csv": retention_matrix.reset_index(),
}

for file_name, dataframe in business_outputs.items():
    output_path = PROCESSED_DATA_DIR / file_name
    dataframe.to_csv(output_path, index=False)
    print(f"Created: {output_path.relative_to(PROJECT_ROOT)}")

## 14. Create a Markdown business report

In [ ]:
def dataframe_to_markdown_table(dataframe: pd.DataFrame, max_rows: int = 15) -> str:
    table = dataframe.head(max_rows).copy()
    if table.empty:
        return "_No records._"
    columns = [str(column) for column in table.columns]
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in table.iterrows():
        values = [str(row[column]) for column in table.columns]
        rows.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator, *rows])

report_path = REPORTS_DIR / "business_analysis_report.md"
report_content = f"""# Business Analysis Report

## Project

Brazilian E-Commerce Analytics Portfolio

## Business KPI Summary

{dataframe_to_markdown_table(business_kpi_summary)}

## Monthly Performance

{dataframe_to_markdown_table(monthly_performance)}

## Top Customer States by Payment Value

{dataframe_to_markdown_table(revenue_by_state)}

## Top Product Categories by Item Revenue

{dataframe_to_markdown_table(category_revenue)}

## Repeat-Customer Summary

{dataframe_to_markdown_table(repeat_customer_summary)}

## Interpretation Notes

- `total_payment_value` represents customer-paid transaction value in the dataset.
- `total_item_value` represents item price value before freight.
- Repeat customers are identified using `customer_unique_id`.
- The retention matrix measures repeat purchasing behavior by monthly acquisition cohort.
- This report is descriptive and does not establish causal relationships.
"""
report_path.write_text(report_content, encoding="utf-8")
print(f"Created report: {report_path.relative_to(PROJECT_ROOT)}")

## Revenue and retention summary

This notebook created order-level and customer-level analytical tables, calculated business KPIs, analyzed revenue by time, state, category, seller, and payment type, measured repeat-customer behavior, created customer cohorts, and exported processed datasets, figures, and a Markdown business report.

### Next notebook

`04_statistical_analysis.ipynb` evaluates statistical relationships such as late delivery versus review score, delivery delay versus review score, payment value versus installments, and customer repeat status versus total customer value.